# 📚 Unity Catalog & Access Control - Complete Guide

This notebook covers three critical Databricks concepts:

## Topics Covered

1. **Unity Catalog** - Unified governance solution for data and AI
2. **Access Control** - Fine-grained permissions and security
3. **Control Plane vs. Data Plane** - Databricks architecture fundamentals

---

## Navigation

* [Unity Catalog Overview](#cell-2)
* [Unity Catalog Architecture](#cell-3)
* [Unity Catalog Code Examples](#cell-4)
* [Access Control Overview](#cell-7)
* [Granting Permissions](#cell-8)
* [Control Plane vs Data Plane](#cell-11)

---

**Prerequisites**: Basic SQL and Python knowledge

**Compute**: Works with any Databricks cluster or SQL Warehouse

## 🏛️ Unity Catalog Overview

**Unity Catalog** is Databricks' unified governance solution for data and AI assets across clouds.

### Key Features

* **Three-level namespace**: `catalog.schema.table`
* **Unified governance**: Manage data, ML models, notebooks, files, and more
* **Fine-grained access control**: Column-level, row-level, and table-level permissions
* **Data lineage**: Track data flow across tables and notebooks
* **Audit logging**: Track all access and changes
* **Data sharing**: Securely share data with Delta Sharing
* **PII protection**: Built-in column masking and row filtering

### Three-Level Namespace

```
catalog               (e.g., production, dev, analytics)
  └── schema          (e.g., sales, customers, inventory)
       └── table      (e.g., orders, users, products)
```

### Benefits

* **Centralized governance** across all workspaces
* **Consistent security** policies
* **Cross-workspace** data sharing
* **Automated lineage** tracking
* **Compliance** with regulations (GDPR, HIPAA, etc.)

## 🏭 Unity Catalog Architecture

### Metastore

The **metastore** is the top-level container for metadata in Unity Catalog.

* One metastore per region
* Can be attached to multiple workspaces
* Contains all catalogs, schemas, and tables
* Stores metadata in cloud storage (S3, ADLS, GCS)

### Hierarchy

```
Metastore (region-level)
  ├── Catalog: production
  │    ├── Schema: sales
  │    │    ├── Table: orders
  │    │    └── Table: customers
  │    └── Schema: marketing
  │         └── Table: campaigns
  ├── Catalog: dev
  │    └── Schema: experiments
  └── Catalog: analytics
       └── Schema: reporting
```

### Security Model

* **Account admins**: Manage metastore and account-level settings
* **Metastore admins**: Manage catalogs and top-level permissions
* **Catalog owners**: Manage schemas within their catalog
* **Schema owners**: Manage tables within their schema
* **Grant-based access**: Explicit permissions required for data access

## 💻 Unity Catalog Code Examples

Let's create Unity Catalog objects and see how to work with them.

In [0]:
%sql
-- Create a new catalog
CREATE CATALOG IF NOT EXISTS my_catalog
COMMENT 'Demo catalog for Unity Catalog examples';

-- Use the catalog
USE CATALOG my_catalog;

-- Create schemas
CREATE SCHEMA IF NOT EXISTS sales
COMMENT 'Sales data and customer information';

CREATE SCHEMA IF NOT EXISTS analytics
COMMENT 'Analytics and reporting tables';

-- Show all catalogs
SHOW CATALOGS;

In [0]:
%sql
-- Create a customers table with PII data
USE CATALOG my_catalog;
USE SCHEMA sales;

CREATE OR REPLACE TABLE customers (
  customer_id BIGINT COMMENT 'Unique customer identifier',
  email STRING COMMENT 'Email address (PII)',
  phone STRING COMMENT 'Phone number (PII)',
  ssn STRING COMMENT 'Social Security Number (PII - highly sensitive)',
  first_name STRING COMMENT 'First name (PII)',
  last_name STRING COMMENT 'Last name (PII)',
  address STRING COMMENT 'Street address (PII)',
  city STRING,
  state STRING,
  zip_code STRING,
  date_of_birth DATE COMMENT 'Date of birth (PII)',
  credit_score INT,
  account_balance DECIMAL(10,2),
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
COMMENT 'Customer data with PII fields';

-- Insert sample data
INSERT INTO customers VALUES
  (1, 'john.doe@email.com', '555-0101', '123-45-6789', 'John', 'Doe', '123 Main St', 'Seattle', 'WA', '98101', '1985-03-15', 720, 15000.50, current_timestamp(), current_timestamp()),
  (2, 'jane.smith@email.com', '555-0102', '987-65-4321', 'Jane', 'Smith', '456 Oak Ave', 'Portland', 'OR', '97201', '1990-07-22', 780, 25000.00, current_timestamp(), current_timestamp()),
  (3, 'bob.johnson@email.com', '555-0103', '456-78-9012', 'Bob', 'Johnson', '789 Pine Rd', 'San Francisco', 'CA', '94102', '1982-11-30', 650, 8500.75, current_timestamp(), current_timestamp());

SELECT * FROM customers;

## 🔒 PII Protection - Column Masking

**Column masking** allows you to protect sensitive data by transforming values based on user permissions.

### Masking Functions

* **MASK()**: Replaces characters with 'X' or other mask character
* **MASK_HASH()**: Returns hash of the value
* **Custom functions**: Create your own masking logic

### Use Cases

* Mask SSN: `XXX-XX-6789` (show last 4 digits)
* Mask email: `j***@email.com`
* Mask phone: `555-****`
* Hash sensitive fields for analytics teams

In [0]:
%sql
-- Create a mask function to show only last 4 digits of SSN
CREATE OR REPLACE FUNCTION my_catalog.sales.mask_ssn(ssn STRING)
RETURNS STRING
RETURN CONCAT('XXX-XX-', RIGHT(ssn, 4));

-- Create a mask function for email
CREATE OR REPLACE FUNCTION my_catalog.sales.mask_email(email STRING)
RETURNS STRING
RETURN CONCAT(LEFT(email, 1), '***@', SPLIT(email, '@')[1]);

-- Create a mask function for phone
CREATE OR REPLACE FUNCTION my_catalog.sales.mask_phone(phone STRING)
RETURNS STRING
RETURN CONCAT(LEFT(phone, 4), '****');

-- Apply column masks to the customers table
ALTER TABLE my_catalog.sales.customers 
ALTER COLUMN ssn SET MASK my_catalog.sales.mask_ssn;

ALTER TABLE my_catalog.sales.customers 
ALTER COLUMN email SET MASK my_catalog.sales.mask_email;

ALTER TABLE my_catalog.sales.customers 
ALTER COLUMN phone SET MASK my_catalog.sales.mask_phone;

-- Now when users without UNMASK privilege query, they see masked data
SELECT customer_id, email, phone, ssn, first_name, last_name 
FROM my_catalog.sales.customers;

## 🔐 Access Control Overview

**Access control** in Unity Catalog provides fine-grained permissions at multiple levels.

### Permission Levels

1. **Catalog-level**: Controls access to entire catalog
2. **Schema-level**: Controls access to schemas within a catalog
3. **Table-level**: Controls access to specific tables
4. **Column-level**: Controls access to specific columns (via masking)
5. **Row-level**: Controls access to specific rows (via row filters)

### Common Privileges

* **USE CATALOG**: Required to access a catalog
* **USE SCHEMA**: Required to access a schema
* **SELECT**: Read data from tables
* **INSERT**: Add new rows
* **UPDATE**: Modify existing rows
* **DELETE**: Remove rows
* **MODIFY**: Change table structure
* **READ FILES**: Read underlying data files
* **WRITE FILES**: Write to underlying data files
* **CREATE**: Create new objects (schemas, tables, etc.)
* **OWNER**: Full control over an object

## ✅ Granting Permissions - Examples

Let's see how to grant different types of permissions to users and groups.

In [0]:
%sql
-- First, let's check who has access to the catalog already
SHOW GRANTS ON CATALOG my_catalog;

-- Grant USE CATALOG to yourself (replace with your actual email)
GRANT USE CATALOG ON CATALOG my_catalog TO `aralokesh@gmail.com`;

-- Grant USE SCHEMA to yourself
GRANT USE SCHEMA ON SCHEMA my_catalog.sales TO `aralokesh@gmail.com`;

-- Grant SELECT on all tables in a schema
GRANT SELECT ON SCHEMA my_catalog.sales TO `aralokesh@gmail.com`;

-- To grant to a GROUP (if you have groups set up):
-- GRANT USE SCHEMA ON SCHEMA my_catalog.sales TO `your_group_name`;

-- Grant CREATE privileges (optional)
GRANT CREATE TABLE ON SCHEMA my_catalog.sales TO `aralokesh@gmail.com`;
GRANT CREATE SCHEMA ON CATALOG my_catalog TO `aralokesh@gmail.com`;

-- Verify grants
SHOW GRANTS ON CATALOG my_catalog;
SHOW GRANTS ON SCHEMA my_catalog.sales;

In [0]:
%sql
-- Grant SELECT on specific table to yourself
GRANT SELECT ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- Grant INSERT and UPDATE to yourself
GRANT INSERT, UPDATE ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- Grant MODIFY (allows schema changes)
GRANT MODIFY ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- For production: Grant to specific groups instead
-- GRANT SELECT ON TABLE my_catalog.sales.customers TO `data_analysts`;
-- GRANT INSERT, UPDATE ON TABLE my_catalog.sales.customers TO `data_engineers`;

-- Show all grants on the table
SHOW GRANTS ON TABLE my_catalog.sales.customers;

In [0]:
%sql
-- By default, you'll see MASKED data due to the column masks we set earlier
-- To see unmasked PII data, you need UNMASK privilege

-- Grant yourself UNMASK privilege to see unmasked data
GRANT SELECT, UNMASK ON TABLE my_catalog.sales.customers TO `aralokesh@gmail.com`;

-- For production: Grant UNMASK only to authorized roles
-- GRANT SELECT, UNMASK ON TABLE my_catalog.sales.customers TO `compliance_team`;

-- Grant UNMASK for specific columns only
-- GRANT UNMASK ON TABLE my_catalog.sales.customers 
--   COLUMNS (ssn, email) 
--   TO `security_team`;

-- To revoke UNMASK:
-- REVOKE UNMASK ON TABLE my_catalog.sales.customers FROM `aralokesh@gmail.com`;

-- Query to see if data is masked or unmasked
SELECT customer_id, email, phone, ssn, first_name, last_name
FROM my_catalog.sales.customers;

## 🛡️ Row-Level Security

**Row filters** allow you to restrict which rows users can see based on conditions.

### Use Cases

* Regional managers see only their region's data
* Sales reps see only their own customers
* Multi-tenant applications with data isolation
* Compliance requirements (e.g., EU users see only EU data)

In [0]:
%sql
-- Create a row filter function that filters by state
CREATE OR REPLACE FUNCTION my_catalog.sales.filter_by_state(state STRING)
RETURNS BOOLEAN
RETURN 
  CASE 
    WHEN IS_ACCOUNT_GROUP_MEMBER('west_coast_team') THEN state IN ('CA', 'OR', 'WA')
    WHEN IS_ACCOUNT_GROUP_MEMBER('east_coast_team') THEN state IN ('NY', 'MA', 'PA')
    WHEN IS_ACCOUNT_GROUP_MEMBER('admin') THEN TRUE
    ELSE FALSE
  END;

-- Apply row filter to customers table
ALTER TABLE my_catalog.sales.customers
SET ROW FILTER my_catalog.sales.filter_by_state ON (state);

-- Now users will only see rows they're authorized to see
SELECT customer_id, first_name, last_name, city, state
FROM my_catalog.sales.customers;

-- To remove row filter:
-- ALTER TABLE my_catalog.sales.customers DROP ROW FILTER;

## 🔄 CDC (Change Data Capture) Overview

**Change Data Capture (CDC)** is a design pattern that tracks changes in source data and propagates them to target systems.

### What is CDC?

CDC captures:
* **INSERT**: New rows added
* **UPDATE**: Existing rows modified
* **DELETE**: Rows removed

Instead of full data loads, CDC only processes **changed data**, making pipelines more efficient.

### Why Use CDC?

* **Efficiency**: Process only changes, not full datasets
* **Real-time**: Near real-time data synchronization
* **History tracking**: Maintain complete audit trail
* **Reduced load**: Less compute and storage for incremental updates
* **Data freshness**: Keep downstream systems up-to-date

### CDC Use Cases

* Sync operational databases to data warehouses
* Build Type 2 Slowly Changing Dimensions (SCD Type 2)
* Create audit logs and data lineage
* Real-time analytics dashboards
* Data replication across systems

## 🏛️ CDC Architecture in Databricks

### CDC Approaches in Databricks

1. **Delta Lake MERGE**: Manual CDC with MERGE operations
2. **Auto Loader CDC**: Streaming CDC from cloud storage
3. **Lakeflow Pipelines CDC**: Declarative CDC with APPLY CHANGES INTO
4. **Debezium/Kafka**: External CDC tools streaming to Databricks

### Delta Lake Change Data Feed

Delta Lake tracks all changes with **Change Data Feed (CDF)**:

```
ENABLE CHANGE DATA FEED on table
↓
Delta automatically tracks:
- _change_type: insert, update_preimage, update_postimage, delete
- _commit_version: Version number
- _commit_timestamp: When change occurred
```

### CDC Pipeline Flow

```
Source System (e.g., MySQL)
  ↓
CDC Capture (Debezium/Binary Logs)
  ↓
Message Queue (Kafka/Kinesis)
  ↓
Auto Loader (Databricks)
  ↓
Bronze Table (Raw CDC events)
  ↓
Silver Table (Merged/Applied changes)
  ↓
Gold Table (Business logic applied)
```

## 💻 CDC Code Examples

Let's implement CDC patterns in Databricks.

In [0]:
%sql
-- Enable Change Data Feed on existing table
ALTER TABLE my_catalog.sales.customers 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Create new table with CDF enabled
CREATE TABLE my_catalog.sales.orders (
  order_id BIGINT,
  customer_id BIGINT,
  order_date DATE,
  total_amount DECIMAL(10,2),
  status STRING
)
TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Insert some test data
INSERT INTO my_catalog.sales.orders VALUES
  (1001, 1, '2024-01-15', 150.00, 'completed'),
  (1002, 2, '2024-01-16', 275.50, 'pending'),
  (1003, 3, '2024-01-17', 89.99, 'completed');

SELECT * FROM my_catalog.sales.orders;

In [0]:
%sql
-- Read all changes since version 0
SELECT *
FROM table_changes('my_catalog.sales.orders', 0);

-- Make some changes to generate CDC events
UPDATE my_catalog.sales.orders 
SET status = 'shipped' 
WHERE order_id = 1002;

INSERT INTO my_catalog.sales.orders VALUES
  (1004, 1, '2024-01-18', 320.00, 'pending');

DELETE FROM my_catalog.sales.orders 
WHERE order_id = 1003;

-- Now read changes again to see CDC events
SELECT 
  order_id,
  customer_id,
  status,
  _change_type,
  _commit_version,
  _commit_timestamp
FROM table_changes('my_catalog.sales.orders', 1)
ORDER BY _commit_version, _commit_timestamp;

In [0]:
# Create a source table with changes
source_changes = spark.createDataFrame([
    (1001, 1, '2024-01-15', 150.00, 'delivered', 'UPDATE'),
    (1002, 2, '2024-01-16', 275.50, 'shipped', 'UPDATE'),
    (1005, 2, '2024-01-19', 450.00, 'pending', 'INSERT'),
    (1004, 1, '2024-01-18', 320.00, 'cancelled', 'UPDATE')
], ['order_id', 'customer_id', 'order_date', 'total_amount', 'status', 'operation'])

# Convert date string to date type
from pyspark.sql.functions import col, to_date, current_timestamp

source_changes = source_changes.withColumn('order_date', to_date(col('order_date')))

# Apply CDC changes using MERGE
source_changes.createOrReplaceTempView('source_changes')

spark.sql("""
  MERGE INTO my_catalog.sales.orders AS target
  USING source_changes AS source
  ON target.order_id = source.order_id
  WHEN MATCHED AND source.operation = 'UPDATE' THEN
    UPDATE SET
      target.status = source.status,
      target.total_amount = source.total_amount
  WHEN MATCHED AND source.operation = 'DELETE' THEN
    DELETE
  WHEN NOT MATCHED AND source.operation = 'INSERT' THEN
    INSERT (order_id, customer_id, order_date, total_amount, status)
    VALUES (source.order_id, source.customer_id, source.order_date, source.total_amount, source.status)
""")

print("CDC changes applied successfully!")

# Verify the results
display(spark.sql("SELECT * FROM my_catalog.sales.orders ORDER BY order_id"))

In [0]:
# Create SCD Type 2 table for customer history
spark.sql("""
  CREATE OR REPLACE TABLE my_catalog.sales.customers_history (
    customer_id BIGINT,
    email STRING,
    phone STRING,
    first_name STRING,
    last_name STRING,
    city STRING,
    state STRING,
    credit_score INT,
    account_balance DECIMAL(10,2),
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
  )
  TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# Initial load - mark all records as current
spark.sql("""
  INSERT INTO my_catalog.sales.customers_history
  SELECT 
    customer_id,
    email,
    phone,
    first_name,
    last_name,
    city,
    state,
    credit_score,
    account_balance,
    created_at as start_date,
    NULL as end_date,
    true as is_current
  FROM my_catalog.sales.customers
""")

print("SCD Type 2 table initialized")

# Simulate a change - customer moved to a new city
spark.sql("""
  UPDATE my_catalog.sales.customers
  SET city = 'Austin', state = 'TX', updated_at = current_timestamp()
  WHERE customer_id = 1
""")

print("Customer data updated - now let's apply SCD Type 2 logic")

In [0]:
# Read changes from source table
changes_df = spark.sql("""
  SELECT 
    customer_id,
    email,
    phone,
    first_name,
    last_name,
    city,
    state,
    credit_score,
    account_balance,
    updated_at
  FROM my_catalog.sales.customers
  WHERE updated_at > (SELECT MAX(start_date) FROM my_catalog.sales.customers_history)
""")

if changes_df.count() > 0:
    changes_df.createOrReplaceTempView('customer_changes')
    
    # Step 1: Close out old records (set end_date and is_current = false)
    spark.sql("""
      MERGE INTO my_catalog.sales.customers_history AS target
      USING customer_changes AS source
      ON target.customer_id = source.customer_id 
         AND target.is_current = true
      WHEN MATCHED THEN
        UPDATE SET
          target.end_date = source.updated_at,
          target.is_current = false
    """)
    
    # Step 2: Insert new records
    spark.sql("""
      INSERT INTO my_catalog.sales.customers_history
      SELECT 
        customer_id,
        email,
        phone,
        first_name,
        last_name,
        city,
        state,
        credit_score,
        account_balance,
        updated_at as start_date,
        NULL as end_date,
        true as is_current
      FROM customer_changes
    """)
    
    print(f"Applied SCD Type 2 changes for {changes_df.count()} customers")
else:
    print("No changes detected")

# View customer history - see the change over time
display(spark.sql("""
  SELECT 
    customer_id,
    first_name,
    last_name,
    city,
    state,
    start_date,
    end_date,
    is_current
  FROM my_catalog.sales.customers_history
  WHERE customer_id = 1
  ORDER BY start_date
"""))

## 🌊 CDC with Lakeflow Pipelines

**Lakeflow Spark Declarative Pipelines** (formerly DLT) provide declarative CDC with `APPLY CHANGES INTO`.

### Benefits

* Declarative syntax - no manual MERGE logic
* Automatic SCD Type 1 or Type 2
* Built-in data quality checks
* Automatic retry and error handling
* Lineage tracking

### Example Pipeline Code

```python
import dlt
from pyspark.sql.functions import col, expr

# Bronze layer - raw CDC events
@dlt.table(
  comment="Raw CDC events from source system",
  table_properties={"quality": "bronze"}
)
def orders_cdc_bronze():
  return (
    spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .load("/mnt/source/cdc/orders/")
  )

# Silver layer - apply CDC changes
dlt.create_streaming_table("orders_silver")

dlt.apply_changes(
  target = "orders_silver",
  source = "orders_cdc_bronze",
  keys = ["order_id"],
  sequence_by = col("updated_timestamp"),
  stored_as_scd_type = 1,  # Use 2 for SCD Type 2
  except_column_list = ["operation", "_metadata"]
)
```

### Key Parameters

* **target**: Destination table name
* **source**: Source table or query
* **keys**: Primary key columns
* **sequence_by**: Column to determine order of changes
* **stored_as_scd_type**: 1 for current state, 2 for history
* **except_column_list**: Columns to exclude from target

## ✈️ Control Plane vs Data Plane

**Databricks architecture** is split into two main components:

### Control Plane

The **control plane** manages workspace configuration and metadata.

**Located in**: Databricks-managed cloud account

**Responsibilities**:
* Workspace UI and notebooks
* Cluster management (create, start, stop, delete)
* Job scheduling and orchestration
* Notebook and code versioning
* Access control and user management
* Audit logs and monitoring
* Unity Catalog metastore

**Components**:
* Web application (workspace UI)
* REST APIs
* Job scheduler
* Cluster manager
* Authentication service

### Data Plane

The **data plane** processes and stores actual data.

**Located in**: Customer's cloud account (your AWS/Azure/GCP)

**Responsibilities**:
* Compute clusters (VMs/instances)
* Data processing and query execution
* Data storage (Delta Lake tables)
* Network connections to data sources
* Actual workload execution

**Components**:
* Spark clusters
* Storage (S3, ADLS, GCS)
* Delta Lake
* ML model training infrastructure

## 🏛️ Architecture Diagram

```
┌────────────────────────────────────────────────┐
│         CONTROL PLANE (Databricks Cloud)            │
│────────────────────────────────────────────────│
│                                                  │
│  💻 Workspace UI    📝 Notebooks              │
│  🛠️  Cluster Manager  📅 Job Scheduler          │
│  🔐 Access Control   📊 Monitoring              │
│  🏛️  Unity Catalog    🔍 Audit Logs              │
│                                                  │
└──────────────────────┬─────────────────────────┘
                       │
                       │ Secure API calls
                       │
┌──────────────────────┴─────────────────────────┐
│        DATA PLANE (Your Cloud Account)             │
│────────────────────────────────────────────────│
│                                                  │
│  ⚙️  Spark Clusters (EC2/VMs)                   │
│  💾 Data Storage (S3/ADLS/GCS)                 │
│  📦 Delta Lake Tables                          │
│  🌐 Network & VPC Configuration                │
│  🔒 Data Encryption (at rest & in transit)     │
│                                                  │
└────────────────────────────────────────────────┘
```

### Key Benefits

* **Security**: Your data never leaves your cloud account
* **Compliance**: Data residency requirements easily met
* **Performance**: Data processing happens close to storage
* **Scalability**: Databricks manages control plane; you scale data plane
* **Cost efficiency**: You only pay for compute and storage you use

## 🔄 Data Flow Example

Let's see how control plane and data plane work together:

### Scenario: Running a Notebook Query

1. **User action** (Control Plane):
   * User clicks "Run" on a notebook cell in Workspace UI
   * Request goes to control plane REST API

2. **Cluster check** (Control Plane):
   * Control plane checks if cluster is running
   * If not, sends command to start cluster in data plane

3. **Code submission** (Control Plane → Data Plane):
   * Control plane sends code to cluster driver in data plane
   * Authentication and authorization checked via Unity Catalog

4. **Query execution** (Data Plane):
   * Spark driver creates execution plan
   * Tasks distributed to worker nodes
   * Data read from S3/ADLS/GCS
   * Computation happens on cluster

5. **Results collection** (Data Plane → Control Plane):
   * First 1000 rows sent back to control plane
   * Results displayed in notebook UI
   * Full results remain in data plane

6. **Monitoring** (Both planes):
   * Control plane logs the job execution
   * Data plane metrics sent to control plane for display
   * Unity Catalog records lineage and access logs

In [0]:
# Check which cloud provider and region your data plane is in
import requests
import json

# Get cluster information
try:
    # Get current cluster details
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    
    print("=" * 60)
    print("DATA PLANE INFORMATION")
    print("=" * 60)
    print(f"\nWorkspace URL: {workspace_url}")
    print(f"Cluster ID: {cluster_id}")
    
    # Get cloud provider from tags
    cloud_provider = spark.conf.get("spark.databricks.cloudProvider", "Unknown")
    print(f"\nCloud Provider: {cloud_provider}")
    
    # Get region information
    try:
        region = spark.conf.get("spark.databricks.clusterUsageTags.region", "Unknown")
        print(f"Region: {region}")
    except:
        print("Region: Unknown")
    
    print("\n" + "=" * 60)
    print("CONTROL PLANE")
    print("=" * 60)
    print("\nLocation: Databricks-managed cloud account")
    print("Access via: https://" + workspace_url)
    
    print("\n" + "=" * 60)
    print("\n✅ Your data is processed in YOUR cloud account (Data Plane)")
    print("✅ Your workspace UI is served from Databricks cloud (Control Plane)")
    
except Exception as e:
    print(f"Unable to retrieve cluster information: {e}")
    print("\nThis is normal if running on serverless compute.")

## 🎓 Summary and Best Practices

### Unity Catalog Best Practices

1. **Organize by environment**: Use separate catalogs for `dev`, `staging`, `prod`
2. **Schema naming**: Group related tables in schemas by domain (e.g., `sales`, `marketing`, `finance`)
3. **Document everything**: Add comments to catalogs, schemas, tables, and columns
4. **Enable CDF early**: Turn on Change Data Feed for tables that need audit trails
5. **Use qualified names**: Always use `catalog.schema.table` to avoid ambiguity

### Access Control Best Practices

1. **Principle of least privilege**: Grant only necessary permissions
2. **Use groups, not individuals**: Manage permissions via groups for easier maintenance
3. **Regular audits**: Review SHOW GRANTS periodically
4. **Protect PII**: Apply column masks and row filters for sensitive data
5. **Document access patterns**: Comment on why specific permissions were granted
6. **Separate duties**: Different groups for read (analysts) vs write (engineers)

### CDC Best Practices

1. **Choose the right approach**: 
   * Simple updates: Use MERGE
   * Complex workflows: Use Lakeflow Pipelines
   * Real-time: Use Auto Loader with CDC

2. **Enable CDF on source tables**: Track all changes automatically
3. **Use sequence columns**: Always have a way to order changes (timestamp, version)
4. **Handle late arrivals**: Design for out-of-order data
5. **Test with deletes**: Ensure your CDC handles all operation types

### Architecture Best Practices

1. **Understand separation**: Know what runs where (control vs data plane)
2. **Secure your data plane**: Use VPC, private links, encryption
3. **Monitor both planes**: Watch metrics from both control and data plane
4. **Cost optimization**: Terminate idle clusters (data plane costs)
5. **Network planning**: Ensure data plane can reach your data sources

---

## 🎉 Congratulations!

You now understand:
* ✅ Unity Catalog hierarchy and governance
* ✅ Fine-grained access control with PII protection
* ✅ CDC patterns for incremental data processing
* ✅ Databricks control plane vs data plane architecture

**Next steps**: Run the code cells to see these concepts in action!